In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
dnascreen_run='_validation_only'

In [3]:
dir='/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/VarSeq_CNV/varseq_tables_on_real_data/run' + dnascreen_run

In [4]:
cnvcalls_path = os.path.join(dir, 'CNV_run' + dnascreen_run + '.tsv')
sample_path = os.path.join(dir, 'Varseq_sample_CNV_run' + dnascreen_run + '.tsv')

cnvcalls_df = pd.read_csv(cnvcalls_path, sep='\t')
sample_df = pd.read_csv(sample_path, sep = '\t')

cov_sample_dir = os.path.join(dir, 'cov_samples')

## The code below is a sanity check of whether samples in reference set are unique and not repeated. Also checks if the target sample is in the reference set itself. 

In [5]:
# Initialize counters and lists to store row indices
duplicate_rows = []
matching_rows = []

# Process each row in sample_df
for index, row in sample_df.iterrows():
    # Get main sample ID without prefix 'cn#_' and suffix '.sorted'
    main_sample_id = re.sub(r'^(cn\d+_)?|\.sorted$', '', row['Samples'])

    # Initialize lists to hold processed reference samples
    sample_names = []

    # Split the reference samples by commas
    for item in row['Reference Samples'].split(','):
        # Extract the sample name using regex
        name_match = re.search(r'^[^\(]+', item)
        
        if name_match:
            # Clean the reference sample name by removing '.hq.sorted.marked' and '.sorted'
            ref_sample_id = re.sub(r'\.hq\.sorted\.marked|\.sorted$', '', name_match.group().strip())
            sample_names.append(ref_sample_id)

    # Check for duplicate reference samples
    if len(sample_names) != len(set(sample_names)):
        duplicate_rows.append(index)  # Store the index of the row with duplicates

    # Check if any reference sample matches the main sample ID
    if main_sample_id in sample_names:
        matching_rows.append(index)  # Store the index of the row with matches

# Print the results
if duplicate_rows:
    print(f"Number of rows with duplicate reference samples: {len(duplicate_rows)}")
    print(f"Rows with duplicates: {duplicate_rows}")
else:
    print("No rows have duplicate reference samples.")

if matching_rows:
    print(f"Number of rows with reference samples matching main sample ID: {len(matching_rows)}")
    print(f"Rows with matches: {matching_rows}")
else:
    print("No rows have reference samples matching main sample ID.")

No rows have duplicate reference samples.
No rows have reference samples matching main sample ID.


## The code below extracts CNV calls per sample.

In [20]:
sample_lst = []
percent_diff_lst = []
avg_noncnv_exons_z_score_lst = []
CNV_coord_chr = []
CNV_coord_start = []
CNV_coord_end = []
region_size_lst = []
number_of_exons_varseq = []
gene_lst = []
condition_lst = []
clin_sig_lst = []
review_status_lst = []
type_lst = []
flag_lst = []
avg_target_mean_depth_lst = []
avg_z_score_lst = []
avg_ratio_lst = []
variants_considered_lst = []
supporting_LOH_variants_lst = []
estimated_CN_lst = []
GC_content_lst = []
p_val_lst = []

# Iterate over each sample
for i in range(31, len(cnvcalls_df.columns), 11): # change upper limit from 11 to 12 when you use settings 'Call at all Precision Levels'
    sample_id = cnvcalls_df.columns[i][:-10]
    percent_diff = sample_df[sample_df['Samples'] == sample_id]['Percent Difference'].values[0]
    
    #loading coverage statistics
    cov_sample_file = os.path.join(cov_sample_dir, f'CNV_run{dnascreen_run} - {sample_id}.tsv')
    cov_sample_df = pd.read_csv(cov_sample_file, sep='\t')

    # these are metrics for CNVs called in VarSeq
    # Check if any p-value is not NaN and <= 0.05
    any_value_check = cnvcalls_df.iloc[0:len(cnvcalls_df), i+10].apply(lambda x: not pd.isnull(x) and (float(x) < 0.05)) #checking for each CNV row if there is CNV.

    # RECORDING CNV CALL METRICS (OR METRICS OF REGIONS NOT CALLED FOR CNV)
    
    # Extract chromosome, start, and end from the Region column
    cov_sample_df[['chr', 'start', 'end']] = cov_sample_df['Region'].str.extract(r'(\d+):(\d+)-(\d+)')
    cov_sample_df['chr'] = cov_sample_df['chr'].astype(int)
    cov_sample_df['start'] = cov_sample_df['start'].astype(int)
    cov_sample_df['end'] = cov_sample_df['end'].astype(int)
    
    if sum(any_value_check) > 0:
        # Check if any value in the specified range meets the condition
        for v in range(0, len(any_value_check)):
            if any_value_check.iloc[v] == True:

                # Extract chr, start, and end of VS calls
                chr = 'chr'+ cnvcalls_df.iloc[v, 0].split(':')[0]
                start_end = cnvcalls_df.iloc[v, 0].split(':')[1]
                start = int(start_end.split('-')[0])
                end = int(start_end.split('-')[1])

                # Filter rows based on sim_chr, sim_start, and sim_end
                noncnv_exons = pd.DataFrame()
                cnv_exons = pd.DataFrame()
                
                mask = (
                    (cov_sample_df['chr'] == chr) &
                    ((cov_sample_df['start'] <= end) &
                    (cov_sample_df['start'] >= start)) |
                    ((cov_sample_df['end'] <= end) &
                    (cov_sample_df['end'] >= start))
                )
            
                noncnv_exons = pd.concat([noncnv_exons, cov_sample_df[~mask]])
                # Drop the auxiliary columns
                noncnv_exons = noncnv_exons.drop(columns=['chr', 'start', 'end'])
            
                # extract average Z-Score of noncnv_exons
                z_score_noncnv_exons = noncnv_exons[f'Z Score for {sample_id}'].values
                avg_noncnv_exons_z_score = np.mean(z_score_noncnv_exons)

                # appending sample-specific metrics
                sample_lst.append(sample_id)
                percent_diff_lst.append(percent_diff)
                avg_noncnv_exons_z_score_lst.append(avg_noncnv_exons_z_score)
                
                CNV_coord_chr.append(chr)
                CNV_coord_start.append(start)
                CNV_coord_end.append(end)
                region_size_lst.append(cnvcalls_df.iloc[v, 4])
                number_of_exons_varseq.append(cnvcalls_df.iloc[v, 2])
                gene_lst.append(cnvcalls_df.iloc[v, 22])
                condition_lst.append(cnvcalls_df.iloc[v, 24])
                clin_sig_lst.append(cnvcalls_df.iloc[v, 25])
                review_status_lst.append(cnvcalls_df.iloc[v, 26])
                type_lst.append(cnvcalls_df.iloc[v, i])
                flag_lst.append(cnvcalls_df.iloc[v, i+1])
                avg_target_mean_depth_lst.append(cnvcalls_df.iloc[v, i+2])
                avg_z_score_lst.append(cnvcalls_df.iloc[v, i+3])
                avg_ratio_lst.append(cnvcalls_df.iloc[v, i+4])
                variants_considered_lst.append(cnvcalls_df.iloc[v, i+6])
                supporting_LOH_variants_lst.append(cnvcalls_df.iloc[v, i+7])
                estimated_CN_lst.append(cnvcalls_df.iloc[v, i+5])
                GC_content_lst.append(cnvcalls_df.iloc[v, i+9])
                p_val_lst.append(cnvcalls_df.iloc[v, i+10])


In [21]:
df_sample_CNV = pd.DataFrame({'sample':sample_lst, 
                              'Percent Difference':percent_diff_lst,
                              'VS_Call_chr':CNV_coord_chr,
                              'VS_Call_start':CNV_coord_start, 
                              'VS_Call_end':CNV_coord_end, 
                              'Size of CNV': region_size_lst,
                              'Number of Exons by VarSeq call':number_of_exons_varseq,
                              'Gene Names': gene_lst,
                              'Conditions': condition_lst,
                              'Clinical Significance': clin_sig_lst,
                              'Review Status': review_status_lst,
                                'Type of CNV':type_lst,
                              'VarSeq Flags for CNV': flag_lst,
                             'Avg Target Mean Depth of CNV': avg_target_mean_depth_lst,
                              'Avg Z-score of Non-CNV Exons':avg_noncnv_exons_z_score_lst,
                             'Avg Z-Score of CNV': avg_z_score_lst,
                             'Avg Read Ratio of CNV': avg_ratio_lst,
                              'Variants considered in CNV': variants_considered_lst,
                              'Supporting LOH variants in CNV': supporting_LOH_variants_lst,
                             'Estimated CN of CNV': estimated_CN_lst,
                             'GC Content of CNV': GC_content_lst,
                             'p-value of CNV':p_val_lst})

In [22]:
# import pandas as pd

# # List to store sample IDs with CNV
# sample_with_CNV = []
# percent_diff = []
# CNV_coord_lst = []
# region_size = []
# clin_sig = []
# review_status = []
# gene_names = []
# Type_lst = []
# Flag_lst = []
# Avg_target_mean_depth = []
# Avg_z_score = []
# Avg_ratio = []
# Estimated_CN = []
# GC_content = []
# p_val = []

# # Iterate over each sample
# for i in range(31, len(df.columns), 11):
#     # Check if any value in the specified range is not NaN and <= 0.05
#     any_value_check = df.iloc[0:len(df), i+10].apply(lambda x: not pd.isnull(x) and 
#                 (float(x) < 0.05)) #checking for each CNV row if there is CNV.

#     if any(any_value_check):
#         CNV_coord = []
#         type = []
#         # Check if any value in the specified range meets the condition
#         for v in range(len(any_value_check)):
#             if any_value_check.iloc[v] == True:
#                 sample_with_CNV.append(df.columns[i])
#                 percent_diff.append(sample_df[sample_df['Samples'] == \
#                                     df.columns[i][:-10]]['Percent Difference'].iloc[0])
#                 CNV_coord_lst.append(df.iloc[v, 0])
#                 region_size.append(df.iloc[v, 4])
#                 Type_lst.append(df.iloc[v, i])
#                 clin_sig.append(df.iloc[v, 19])
#                 review_status.append(df.iloc[v, 20])
#                 gene_names.append(df.iloc[v, 22])
#                 Flag_lst.append(df.iloc[v, i+1])
#                 Avg_target_mean_depth.append(df.iloc[v, i+2])
#                 Avg_z_score.append(df.iloc[v, i+3])
#                 Avg_ratio.append(df.iloc[v, i+4])
#                 Estimated_CN.append(df.iloc[v, i+5])
#                 GC_content.append(df.iloc[v, i+9])
#                 p_val.append(df.iloc[v, i+10])

In [23]:
# df_sample_CNV = pd.DataFrame({'Sample':sample_with_CNV, 
#                               'Percent Difference': percent_diff,
#                               'CNV_coordinates':CNV_coord_lst, 
#                               'Region size': region_size, 
#                                 'Type':Type_lst, 
#                               'Clinical significance': clin_sig,
#                              'Review status': review_status,
#                              'Gene names': gene_names,
#                               'Flags': Flag_lst,
#                              'Avg Target Mean Depth': Avg_target_mean_depth,
#                              'Avg Z-score': Avg_z_score, 
#                              'Avg Ratio': Avg_ratio,
#                              'Estimated CN': Estimated_CN,
#                              'GC Content': GC_content,
#                              'p-value':p_val})

In [24]:
df_sample_CNV

,sample,Percent Difference,VS_Call_chr,VS_Call_start,VS_Call_end,Size of CNV,Number of Exons by VarSeq call,Gene Names,Conditions,Clinical Significance,...,VarSeq Flags for CNV,Avg Target Mean Depth of CNV,Avg Z-score of Non-CNV Exons,Avg Z-Score of CNV,Avg Read Ratio of CNV,Variants considered in CNV,Supporting LOH variants in CNV,Estimated CN of CNV,GC Content of CNV,p-value of CNV
0,DNS-XTHS-0009-G09-DNS000601_S167,11.13240,chr13,32326076,32326638,563,3,"BRCA2,BRCA2,BRCA2,BRCA2","RCV001370372,RCV000460688,RCV000819896,RCV0005...","Hereditary breast ovarian cancer syndrome,Here...",...,NaN,436.0230,0.226238,-2.67629,0.535723,0.0,0.0,1.0,0.335702,1.954818e-09
1,DNS-XTHS-0010-D04-DNS000653_S220,18.70680,chr1,55039813,55058672,18860,10,"BSND,PCSK9",RCV001379486,"Hypercholesterolemia, autosomal dominant, 3",...,NaN,336.2140,-0.206939,2.56973,1.101460,1.0,0.0,3.0,0.551379,2.152139e-14
2,DNS-XTHS-0010-D04-DNS000653_S220,18.70680,chr2,21037933,21043970,6038,5,NaN,NaN,NaN,...,NaN,283.3620,-0.199822,4.47600,1.297670,2.0,0.0,3.0,0.469361,8.455919e-17
3,DNS-XTHS-0010-D04-DNS000653_S220,18.70680,chr2,47783209,47783518,310,1,"LOC129933707,MSH6,LOC129933706,LOC129933707,MS...","RCV000476296,RCV000708537,RCV000504163,RCV0010...","Lynch syndrome,Hereditary nonpolyposis colorec...",...,"Insufficient Ratio,Extreme GC Content",300.3940,-0.091415,5.29205,1.376070,2.0,0.0,3.0,0.716129,5.860854e-07
4,DNS-XTHS-0015-D09-DNS001168_S356,12.13930,chr1,55058473,55058672,200,1,NaN,NaN,NaN,...,"Deletion Contains Heterozygous Variants,Insuff...",94.7350,0.046745,-4.06386,0.643014,3.0,0.0,1.0,0.630000,1.106449e-05
5,DNS-XTHS-0016-E03-DNS001217_S21,14.96280,chr1,55043818,55063609,19792,11,"BSND,PCSK9",RCV001379486,"Hypercholesterolemia, autosomal dominant, 3",...,NaN,254.5980,-0.244355,1.95199,1.208400,2.0,0.0,3.0,0.560934,1.182129e-11
6,DNS-XTHS-0016-E03-DNS001217_S21,14.96280,chr17,43063849,43063976,128,1,NaN,NaN,NaN,...,Insufficient Ratio,89.5312,-0.079105,-3.35082,0.682034,0.0,0.0,1.0,0.382812,1.673247e-03
7,DNS-XTHS-0017-H02-DNS001307_S112,13.74470,chr1,55039813,55058234,18422,9,"BSND,PCSK9",RCV001379486,"Hypercholesterolemia, autosomal dominant, 3",...,NaN,657.1850,-0.222540,2.19445,1.228270,2.0,0.0,3.0,0.551406,5.585580e-13
8,DNS-XTHS-0017-H02-DNS001307_S112,13.74470,chr2,21040913,21043970,3058,4,NaN,NaN,NaN,...,NaN,403.0520,-0.161640,2.00564,1.377400,0.0,0.0,3.0,0.503270,2.499914e-06
9,DNS-XTHS-0017-H02-DNS001307_S112,13.74470,chr2,47403167,47403427,261,1,MSH2,RCV003122261,Hereditary nonpolyposis colorectal neoplasms,...,Insufficient Ratio,757.6820,-0.129850,3.26206,1.255530,0.0,0.0,3.0,0.670498,2.348819e-04


In [25]:
df_sample_CNV.to_csv(dir + '/CNV_in_each_sample_run'+ dnascreen_run + '.csv')